# NIDAR drone maze: PPO + reflex training (Colab)

Trains the goal-directed PPO policy with the fruit-fly reflex layer in the loop, headless (no GPU, no rendering needed).

**Before you start**, make one zip called `nidar_rl.zip` that contains, at the top level:
`fly_x2_keyboard.py`, `nidar_env.py`, `reflex.py`, `train_ppo.py`, `nidar_arena.xml`, `nidar_arena_layout.json` and the `skydio_x2/` folder (with `x2.xml` and its mesh assets).

On your PC (PowerShell) you can build it with:
```powershell
$root = "C:\Users\HP\Downloads\fruitfly_rl_drone_maze\new_try"
$tmp  = "$root\nidar_rl"
New-Item -ItemType Directory -Force $tmp | Out-Null
Copy-Item "$root\files_ppo\*.py","$root\nidar_sim\fly_x2_keyboard.py","$root\nidar_sim\nidar_arena.xml","$root\nidar_sim\nidar_arena_layout.json" $tmp
Copy-Item "$root\skydio_x2" $tmp -Recurse
Compress-Archive "$tmp\*" "$root\nidar_rl.zip" -Force
```

## 1. Install

In [ ]:
!pip install -q mujoco gymnasium stable-baselines3

## 2. Get your files in
Run **either** 2a (upload) **or** 2b (Google Drive).

In [ ]:
# 2a. Upload nidar_rl.zip from your computer
from google.colab import files
up = files.upload()                      # pick nidar_rl.zip
!rm -rf /content/nidar_rl && mkdir -p /content/nidar_rl
!unzip -q -o nidar_rl.zip -d /content/nidar_rl

In [ ]:
# 2b. Or: from Google Drive (put nidar_rl.zip in My Drive first)
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/nidar_rl && mkdir -p /content/nidar_rl
!unzip -q -o /content/drive/MyDrive/nidar_rl.zip -d /content/nidar_rl

## 3. Check the files

In [ ]:
import os
os.chdir('/content/nidar_rl')
need = ['fly_x2_keyboard.py','nidar_env.py','reflex.py','train_ppo.py',
        'nidar_arena.xml','nidar_arena_layout.json','skydio_x2/x2.xml']
missing = [f for f in need if not os.path.exists(f)]
print('missing:', missing or 'none')
print('CPU cores:', os.cpu_count())
if missing:
    print('If the zip unpacked into a sub-folder, cd into it, e.g. os.chdir("/content/nidar_rl/nidar_rl")')

## 4. Smoke test (about a minute)
If this finishes and prints evaluation lines, the setup is good.

In [ ]:
!python -u train_ppo.py --envs 1 --scale 0.01 --episodes 2 --out runs/smoke

## 5. Full training

Saves `ppo_nidar.zip` and `vecnorm.pkl` after **every stage**. The output goes to Google Drive so a Colab disconnect doesn't lose it (mount Drive in step 2b, or run the next cell).
Total is about 2.2M steps. Change `--scale` (e.g. `0.5`) to shorten it, and `--envs` to the number of CPU cores (Colab free usually has 2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nidar_runs/run1'

In [ ]:
!python -u train_ppo.py --envs 2 --scale 1.0 --out {OUT}

Optional: ablation, training **without** the reflex layer (change `OUT` first so you don't overwrite the main run):

In [ ]:
# OUT2 = '/content/drive/MyDrive/nidar_runs/no_reflex'
# !python -u train_ppo.py --envs 2 --no-reflex --out {OUT2}

## 6. Evaluate a saved run (reflex ON vs OFF)

In [ ]:
!python -u train_ppo.py --eval-only {OUT} --episodes 30

## 7. Download the result

You need **both** files to run the policy locally with `run_policy.py`.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/nidar_run1', 'zip', OUT)
files.download('/content/nidar_run1.zip')

**Run it on your PC:** unzip into `files_ppo\runs\nidar`, then
```powershell
python .\run_policy.py --run runs\nidar --goal 3 --x2 $x2 --arena "$sim\nidar_arena.xml" --layout "$sim\nidar_arena_layout.json"
```